# JOINs - World Bank’s International Education Dataset

## Activity Overview

Now, we’ll write some queries that join multiple tables together.

In this notebook, we’ll work with the <b>World Bank’s International Education Dataset</b> in BigQuery Public Dataset. By using JOIN statements, we’ll be able to fully harness the power of relational databases by combining data from tables linked by keys. 

## Review JOINs
The two most common kinds of JOIN statements are INNER JOINs and OUTER LEFT JOINs (also known simply as LEFT JOINs). As a review:
- INNER JOIN: Returns only the rows where the target appears in both tables. 
- LEFT JOIN: Returns every row from the left table, as well as any rows from the right table with matching keys found in the left table. 

## Load and examine the dataset

In [1]:
from google.cloud import bigquery

print(bigquery.__version__)

3.40.1


In [2]:
client = bigquery.Client()

print(client.project)

myproject001-504709


In [4]:
# This code only tests if the BigQuery connection is on.
query = """
SELECT
  name,
  SUM(number) AS total_births
FROM `bigquery-public-data.usa_names.usa_1910_current`
GROUP BY name
ORDER BY total_births DESC
LIMIT 10
"""

df = client.query(query).to_dataframe()

df.head()

,name,total_births
0,James,5054074
1,John,4910976
2,Robert,4763102
3,Michael,4396482
4,William,3939912


## Queries with JOINS and aliases

Now, it’s time to actually query the dataset. As a starting point, we try a query that pulls information from both the <b>international_education</b> and <b>country_summary</b> tables.

In [5]:
query = """
SELECT 
    `bigquery-public-data.world_bank_intl_education.international_education`.country_name, 
    `bigquery-public-data.world_bank_intl_education.country_summary`.country_code, 
    `bigquery-public-data.world_bank_intl_education.international_education`.value
FROM 
    `bigquery-public-data.world_bank_intl_education.international_education`
INNER JOIN 
    `bigquery-public-data.world_bank_intl_education.country_summary` 
ON `bigquery-public-data.world_bank_intl_education.country_summary`.country_code = `bigquery-public-data.world_bank_intl_education.international_education`.country_code
"""

df = client.query(query).to_dataframe()

df.head()

,country_name,country_code,value
0,Chad,TCD,234686.0
1,Chad,TCD,268384.0
2,Chad,TCD,317174.0
3,Chad,TCD,63307.0
4,Chad,TCD,136113.0


This basic query joins the tables on the country_code foreign key, and returns the country name, country code, and value column.\
This is quite a long, unwieldy query for such a basic result! The length of each table name (which must include the full address for each table for BigQuery to know where to pull the data from) makes this hard to read and work with.  

However, we can solve this by setting an alias for each table.

### Use Descriptive Aliases 

Let's try using descriptive aliases that tell us what they represent. This next query is the same query as the previous one, but with aliases to improve readability.

In [6]:
query = """
SELECT 
    edu.country_name,
    summary.country_code,
    edu.value
FROM 
    `bigquery-public-data.world_bank_intl_education.international_education` AS edu
INNER JOIN 
    `bigquery-public-data.world_bank_intl_education.country_summary` AS summary
ON edu.country_code = summary.country_code
"""

df = client.query(query).to_dataframe()

df.head()

,country_name,country_code,value
0,Chad,TCD,306639.0
1,Chad,TCD,170052.0
2,Chad,TCD,123744.0
3,Chad,TCD,57804.0
4,Chad,TCD,101428.0


This query is much easier to read and understand. Recall that we can set aliases for tables by specifying the alias for the table after the table’s name in FROM and/or JOIN statements. 

For this example, the international_education table was renamed as <b>edu</b>, and the country_summary table as <b>summary</b>. Using descriptive aliases is a best practice and will help us keep our queries clean, readable, and easy to work with. 


### Use a JOIN to Answer a Question

Now we can answer an actual data question using this dataset. \
<b>What is the average amount of money spent per region on education?</b>

In [8]:
query = """
SELECT 
    AVG(edu.value) average_value, summary.region
FROM 
    `bigquery-public-data.world_bank_intl_education.international_education` AS edu
INNER JOIN 
    `bigquery-public-data.world_bank_intl_education.country_summary` AS summary
ON edu.country_code = summary.country_code
WHERE summary.region IS NOT null
GROUP BY summary.region
ORDER BY average_value DESC
"""

df = client.query(query).to_dataframe()

df

,average_value,region
0,4.165344e+10,North America
1,3.882406e+09,East Asia & Pacific
2,2.696535e+09,South Asia
3,2.374149e+09,Europe & Central Asia
4,9.734777e+08,Middle East & North Africa
5,9.659991e+08,Latin America & Caribbean
6,2.116920e+08,Sub-Saharan Africa


In this query, an alias is also set to give the AVG(edu.value) a more descriptive name for the temporary table the query returns.

The WHERE statement also excludes rows with any null information. This is necessary to present the data succinctly and display only seven rows for the seven regions represented in the data. However, this WHERE statement means that the results will return the same regardless of which JOIN we use. 

In the next section, we’ll explore a situation where we need to use a specific kind of join in our query.


## INNER JOINs versus OUTER JOINs

In the last query, we used an INNER JOIN to find the average amount of money spent per region on education. Because of the WHERE statement in this query, using any kind of JOIN produces the same result.

Now, we will write a LEFT JOIN, a type of OUTER JOIN, for a situation where the type of query we use will change the result we return.

<b>Scenario: </b>

We have been tasked to provide data for a feature sports article on Michael Jordan’s basketball career. The writer wants to include a funny twist and asks us to find out if Michael Jordan played better at schools with animal mascots.

To analyze his early career, we start with the years he played basketball in college. We need to examine <b>National Collegiate Athletic Association (NCAA) college basketball</b> stats from 1984.

We’ll need a list of all NCAA Division I colleges and universities; their mascots, if applicable; and their number of wins and losses. We can find this information in the public dataset <b>ncaa_basketball</b> on BigQuery.

Next, we will write a query. Our query should join the season statistics from one table with the mascot information from another. We need to use a LEFT JOIN instead of an INNER JOIN because not all teams have mascots. If we use an INNER JOIN, we would exclude teams with no mascot.



In [9]:
query = """
SELECT
 seasons.market AS university,
 seasons.name AS team_name,
 seasons.wins,
 seasons.losses,
 seasons.ties,
 mascots.mascot AS team_mascot
FROM
 `bigquery-public-data.ncaa_basketball.mbb_historical_teams_seasons` AS seasons
LEFT JOIN
 `bigquery-public-data.ncaa_basketball.mascots` AS mascots
ON
 seasons.team_id = mascots.id
WHERE
 seasons.season = 1984
 AND seasons.division = 1
ORDER BY
 seasons.market
"""

df = client.query(query).to_dataframe()

df

,university,team_name,wins,losses,ties,team_mascot
0,None,None,15,13,0,None
1,None,None,6,20,0,None
2,Alabama State University,Hornets,14,17,0,Hornets
3,Alcorn State University,Braves,23,7,0,Hawk
4,American University,Eagles,9,19,0,Eagle
...,...,...,...,...,...,...
276,Western Michigan University,Broncos,12,16,0,Bronco
277,Wichita State University,Shockers,18,13,0,Wheat
278,Xavier University,Musketeers,16,13,0,Musketeer
279,Yale University,Bulldogs,14,12,0,Bulldog


This is an good demostration of when a LEFT JOIN is more helpful than an INNER JOIN. With this query, we can look at college basketball statistics to get a better sense of Michael Jordan’s early career, find out more information about which teams had mascots, and answer our business question.